# Step 6 — External validation

**Lymphoma AI Research** (`dlbcl-prognosis`) · locked analysis plan v1.0, section 10 · code version 06-1.0

Research project only. Nothing in this notebook is a diagnostic or treatment tool.

Each stage starts only after the previous one has passed:

1. **Integrity checks (sections 2–4).** Plan lock, plan files, amendments, every input file and the frozen model (`model_04.json`) are compared with `plan_lock_04.json`, `manifest_04b.json` and `manifest_05.json`. Nothing is fitted in this notebook.
2. **Gate (section 6).** Only then are the external cohorts opened for the first time: GSE10846 (main) and GSE87371 (supportive). Every opening is logged.
3. **Scoring (section 7).** The 44 frozen genes and coefficients are applied unchanged. Expression is z-scored gene by gene **within each cohort**, on its primary analysis population (plan §10.2).
4. **Metrics (section 8).** H2 (HR per SD of LP > 1, 95% CI excluding 1) and the other metrics of §8.2 except the calibration slope; high/low risk at each cohort's own median LP (§10.3).
5. **Sensitivity analyses (section 9).** GSE87371 without R-mini-CHOP21; correlation outliers excluded in each cohort; both with the scores of the primary analysis (no re-scaling).
6. Figures, red flags, the pre-specified conclusion (§12), outputs, `manifest_06.json` and a summary to paste back (sections 10–12).

TR: Bu not defteri kilitli planın 10. bölümünü uygular. Önce plan kilidi, girdi dosyaları ve dondurulmuş modelin SHA-256 özeti manifestolarla karşılaştırılır. Hepsi geçerse dış kohortlar ilk kez açılır. 44 gen ve katsayıları hiç değişmeden uygulanır; ifade her kohortun kendi içinde z-skorlanır, risk grupları kohortun kendi medyanıyla ayrılır. Bu not defterinde hiçbir model kurulmaz.

## 1. Setup
Mounts Google Drive (in Colab), installs scikit-survival and lifelines if they are missing, and sets the paths. Outside Colab the project folder is taken from the environment variable `DLBCL_PROJECT_DIR`.

TR: Drive'ı bağlar, eksikse scikit-survival ve lifelines kütüphanelerini kurar, klasör yollarını ayarlar. Çalışma süresi toplamda birkaç dakikadır; model kurulmadığı için 5. adım gibi uzun sürmez.

In [ ]:
import importlib, os, subprocess, sys

for pkg, mod in (('scikit-survival', 'sksurv'), ('lifelines', 'lifelines')):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/dlbcl-prognosis'
except ImportError:
    PROJECT_DIR = os.environ['DLBCL_PROJECT_DIR']

import datetime, hashlib, json, math, platform, re, time

import matplotlib
import matplotlib.collections
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
import sksurv
import lifelines
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts
from lifelines.statistics import logrank_test, proportional_hazard_test
from lifelines.utils import median_survival_times
from sksurv.metrics import concordance_index_censored, concordance_index_ipcw
from sksurv.util import Surv

CODE_VERSION = '06-1.0'
RES04, RES05 = f'{PROJECT_DIR}/results/04', f'{PROJECT_DIR}/results/05'
RES, FIG = f'{PROJECT_DIR}/results/06', f'{PROJECT_DIR}/figures/06'
os.makedirs(RES, exist_ok=True)
os.makedirs(FIG, exist_ok=True)
T_START = time.time()
print('Project folder:', PROJECT_DIR)
print(f'Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | '
      f'scikit-survival {sksurv.__version__} | lifelines {lifelines.__version__}')

## 2. Verify the plan lock and the input files
The lock file must be exactly the one recorded after 04a; the two plan files must match the hashes stored in it, and `plan_amendments.md` must have no entries. Every input file is hashed and compared with the lock, and the same hashes must appear in `manifest_04b.json` and `manifest_05.json`. Hashing reads the bytes of a file but does not load it as data: the external cohorts stay closed until section 6.

TR: Kilit dosyasını, plan dosyalarını ve bütün girdi dosyalarını SHA-256 ile kontrol eder; aynı özetlerin 4b ve 5. adım manifestolarında da olması gerekir. Bir şey değiştiyse durur. Dış kohortlar bu aşamada açılmaz.

In [ ]:
EXPECTED_LOCK_SHA256 = '8d87cf887e2db4770ae3273755d4901cd0d5a2a010fdc729159e007aa129bc7d'    # progress_step4a.md

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def _clean(o):
    """JSON-safe copy: NaN/inf -> null, numpy scalars -> Python."""
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.bool_, bool)):
        return bool(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return float(o) if math.isfinite(float(o)) else None
    return o

def write_json(obj, path):
    with open(path, 'w') as f:
        json.dump(_clean(obj), f, indent=2, sort_keys=False, allow_nan=False)

LOCK_PATH = f'{RES04}/plan_lock_04.json'
assert os.path.exists(LOCK_PATH), 'results/04/plan_lock_04.json not found.'
LOCK_SHA = sha256_file(LOCK_PATH)
assert LOCK_SHA == EXPECTED_LOCK_SHA256, (
    f'The lock file is not the one recorded after 04a ({EXPECTED_LOCK_SHA256[:12]} vs {LOCK_SHA[:12]}). Stop.')
with open(LOCK_PATH) as f:
    LOCK = json.load(f)
for kind in ('json', 'md'):
    path = f"{PROJECT_DIR}/{LOCK[f'plan_{kind}_file']}"
    assert sha256_file(path) == LOCK[f'plan_{kind}_sha256'], f'{path} differs from the locked version. Stop.'
with open(f"{PROJECT_DIR}/{LOCK['plan_json_file']}") as f:
    PLAN = json.load(f)
assert PLAN['version'] == '1.0'

AMEND_PATH = f"{PROJECT_DIR}/{PLAN['deviations']['file']}"
amend_text = open(AMEND_PATH).read() if os.path.exists(AMEND_PATH) else ''
AMENDMENTS = re.findall(r'^## Amendment \d+.*$', amend_text, flags=re.M)
if AMENDMENTS:
    raise RuntimeError(f'plan_amendments.md has {len(AMENDMENTS)} entries. This notebook implements plan v1.0 '
                       'without amendments; check before running it.')

with open(f'{RES04}/manifest_04b.json') as f:
    M4 = json.load(f)
with open(f'{RES05}/manifest_05.json') as f:
    M5 = json.load(f)
for name, m in (('manifest_04b', M4), ('manifest_05', M5)):
    assert m['plan']['lock_file_sha256'] == LOCK_SHA, f'{name} was produced under a different plan lock. Stop.'
    assert m['plan']['plan_json_sha256'] == LOCK['plan_json_sha256'] and m['plan']['plan_md_sha256'] == LOCK['plan_md_sha256']
    assert m['plan']['amendments'] == 0

INPUT_SHA = {}
print('Input files (SHA-256 vs lock, manifest_04b, manifest_05):')
for key, rel in LOCK['input_files'].items():
    path = f'{PROJECT_DIR}/{rel}'
    assert os.path.exists(path), f'missing input file: {rel}'
    h = sha256_file(path)
    refs = [LOCK['input_sha256'][key], M4['inputs_verified_against_lock'][key]['sha256'],
            M5['inputs_verified_against_lock'][key]['sha256']]
    status = 'match' if all(h == r for r in refs) else 'CHANGED'
    INPUT_SHA[key] = {'file': rel, 'sha256': h, 'status': status}
    print(f'  {status:8s} {rel}')
CHANGED = [k for k, v in INPUT_SHA.items() if v['status'] != 'match']
assert not CHANGED, f'Input files changed since the plan was locked: {CHANGED}. Stop.'
print(f"Plan v{PLAN['version']} lock verified (lock {LOCK_SHA[:12]}); no amendments; "
      f'all {len(INPUT_SHA)} input files unchanged.')

## 3. Verify the frozen model and the state of step 5
The SHA-256 of `model_04.json` must equal the value recorded in `progress_step4b.md`, in `manifest_04b.json` (two entries) and in `manifest_05.json`. Its content is checked (44 genes, unique Entrez IDs, non-zero coefficients). Step 5 must be complete, and `manifest_05.json` must list both external expression files as *hashed only, never opened*.

TR: Dondurulmuş modelin özeti dört kayıtla karşılaştırılır. 5. adımın bitmiş olduğu ve dış kohortların o adımda açılmadığı manifestodan kontrol edilir.

In [ ]:
EXPECTED_MODEL_SHA256 = '661e2b80be774252a7fd613116ecc06673a8617ed8a2230bdae56170e3908403'   # progress_step4b.md
MODEL_PATH = f'{RES04}/model_04.json'
MODEL_SHA = sha256_file(MODEL_PATH)
HASH_REFS = {'progress_step4b.md': EXPECTED_MODEL_SHA256,
             'manifest_04b.json, frozen_model.sha256': M4['frozen_model']['sha256'],
             'manifest_04b.json, outputs entry': M4['outputs']['results/04/model_04.json'],
             'manifest_05.json, frozen_model.sha256': M5['frozen_model']['sha256']}
for ref, h in HASH_REFS.items():
    assert MODEL_SHA == h, f'model_04.json ({MODEL_SHA[:12]}) does not match {ref} ({h[:12]}). Stop.'

with open(MODEL_PATH) as f:
    MODEL = json.load(f)
assert not MODEL['null_model'], 'The frozen model is empty (null result); there is nothing to validate.'
G = pd.DataFrame(MODEL['genes'])
G['entrez'] = G['entrez'].astype(str)
assert len(G) == M4['frozen_model']['n_genes'] == M5['frozen_model']['n_genes'] == 44
assert G['entrez'].is_unique and (G['coef'] != 0).all() and np.isfinite(G['coef']).all()
assert np.isclose(MODEL['penalty']['lambda'], M4['frozen_model']['lambda'], rtol=1e-12)
COEF = G['coef'].to_numpy(dtype=float)
NESTED_CV_C = M4['nested_cv']['min']['mean_c']
TEST_RESULT = M5['internal_validation']

assert M5.get('conclusion'), 'manifest_05.json has no conclusion; finish step 5 first.'
assert os.path.exists(f'{RES05}/first_results_05.json'), 'Step 5 has no first_results_05.json; finish step 5 first.'
never_opened_5 = set(M5['data_access']['hashed_only_never_opened'].values())
for rel in ('data/processed/expr_GSE10846.parquet', 'data/processed/expr_GSE87371.parquet'):
    assert rel in never_opened_5, f'manifest_05 does not list {rel} as never opened. Stop and check.'

ACCESS_LOG = f'{RES}/external_access_log_06.jsonl'
PREVIOUS_OPENINGS = sum(1 for line in open(ACCESS_LOG) if line.strip()) if os.path.exists(ACCESS_LOG) else 0
FIRST_PATH = f'{RES}/first_results_06.json'
print(f'Frozen model verified: SHA-256 {MODEL_SHA[:12]}… = {len(HASH_REFS)} records; {len(G)} genes, '
      f"lambda {MODEL['penalty']['lambda']:.6f}")
print(f"Step 5: {M5['conclusion']}; test HR per SD {TEST_RESULT['hr_per_sd']:.2f}, C {TEST_RESULT['harrell_c']:.3f}")
print(f'External cohorts opened before this run: {PREVIOUS_OPENINGS} logged opening(s)'
      + ('' if PREVIOUS_OPENINGS == 0 else ' -> this is a re-run; results will be compared with first_results_06.json'))

## 4. Settings from the locked plan
All settings come from `analysis_plan_04.json`. The step-6 text of the plan is compared word for word with what this notebook implements, so the code cannot drift from the plan.

TR: Bütün ayarlar kilitli plandan okunur; plandaki 6. adım metni bu not defterinin uyguladığıyla birebir karşılaştırılır.

In [ ]:
S6, S5, Q, EP = PLAN['step6_external_validation'], PLAN['step5_internal_validation'], PLAN['qc'], PLAN['endpoints']
assert S6['transfer'] == 'frozen Entrez IDs and coefficients; no re-selection, re-fitting or re-tuning'
assert S6['scaling'] == 'gene-wise z-score within each cohort, computed on its primary analysis population'
assert S6['risk_group_cutoff'] == 'median LP of each cohort'
assert S6['metrics'] == 'as step 5, except calibration slope'
assert S6['sensitivity'] == ['GSE87371 without R-mini-CHOP21 patients (n = 13)', 'correlation outliers excluded']
assert S6['sensitivity_scaling'] == 'scores of the primary analysis; no re-scaling'
assert PLAN['conclusion_rules']['externally_validated'] == 'H2 met in GSE10846 and HR > 1 in GSE87371'
assert PLAN['risk_groups']['optimal_cutpoint_search'] is False

TIME, EVENT = EP['primary']['time'], EP['primary']['event']
TAU0, MIN_AT_RISK = float(EP['uno_c_tau_months']), float(EP['uno_c_min_at_risk_fraction'])
N_BOOT = int(S5['bootstrap_resamples'])
BOOT_SEED = int(PLAN['seeds']['validation_bootstrap'])
FLAG_P, FLAG_S = Q['primary_exclusion_flag'], Q['sensitivity_exclusion_flag']
DDOF = int(PLAN['features']['variance_ddof'])
COHORTS = {
    'GSE10846': {'role': 'main', 'plan': PLAN['cohorts']['external_main'],
                 'expected_excl': Q['expected_exclusions']['GSE10846'], 'file': 'data/processed/expr_GSE10846.parquet'},
    'GSE87371': {'role': 'supportive', 'plan': PLAN['cohorts']['external_supportive'],
                 'expected_excl': Q['expected_exclusions']['GSE87371'], 'file': 'data/processed/expr_GSE87371.parquet'},
}
for gse, c in COHORTS.items():
    assert c['plan']['gse'] == gse and c['plan']['first_used_in_step'] == 6
MINI_CHOP_EXPECTED = 13
print(f'tau = {TAU0:g} months (min. {MIN_AT_RISK:.0%} at risk); {N_BOOT} bootstrap resamples, seed {BOOT_SEED}; '
      f'primary QC flag {FLAG_P!r}, sensitivity flag {FLAG_S!r}')

## 5. Validation functions
The same functions as step 5 (Harrell C with bootstrap CI, Uno C with the τ rule, Kaplan–Meier summaries, log-rank, HR per SD, Schoenfeld check), with the calibration slope switched off (plan §10.4).

TR: 5. adımdaki fonksiyonların aynısı; yalnızca kalibrasyon eğimi kapalı.

In [ ]:
def harrell_c(time_, event, score):
    return float(concordance_index_censored(np.asarray(event).astype(bool), np.asarray(time_, dtype=float),
                                            np.asarray(score, dtype=float))[0])

def uno_tau(time_, tau0, min_frac):
    time_ = np.asarray(time_, dtype=float)
    if np.mean(time_ >= tau0) >= min_frac:
        return float(tau0), f'{tau0:g} months (plan value; {np.mean(time_ >= tau0):.0%} at risk)'
    k = int(np.ceil(min_frac * len(time_)))
    tau = float(np.sort(time_)[::-1][k - 1])
    return tau, f'{tau:.1f} months (fewer than {min_frac:.0%} at risk at {tau0:g} months)'

def uno_c(time_, event, score, tau):
    y = Surv.from_arrays(np.asarray(event).astype(bool), np.asarray(time_, dtype=float))
    return float(concordance_index_ipcw(y, y, np.asarray(score, dtype=float), tau=tau)[0])

def cox_uni(time_, event, x, name):
    df = pd.DataFrame({'T': np.asarray(time_, dtype=float), 'E': np.asarray(event, dtype=int), name: np.asarray(x, dtype=float)})
    cph = CoxPHFitter().fit(df, 'T', 'E')
    s = cph.summary.loc[name]
    return ({'hr': float(s['exp(coef)']), 'hr_ci_95': [float(s['exp(coef) lower 95%']), float(s['exp(coef) upper 95%'])],
             'coef': float(s['coef']), 'coef_ci_95': [float(s['coef lower 95%']), float(s['coef upper 95%'])],
             'p_wald': float(s['p'])}, cph, df)

def km_rows(time_, event, labels, order, t_os=60.0):
    """n, deaths, median OS and OS at t_os months (95% CIs, lifelines defaults) for each group."""
    rows = []
    for g in order:
        m = labels == g
        row = {'group': g, 'n': int(m.sum()), 'deaths': int(event[m].sum()), 'median_os_months': None,
               'median_ci_95': [None, None], 'os_5y': None, 'os_5y_ci_95': [None, None]}
        if m.sum() > 0:
            kmf = KaplanMeierFitter().fit(time_[m], event[m])
            med = float(kmf.median_survival_time_)
            mci = median_survival_times(kmf.confidence_interval_).iloc[0].to_numpy(dtype=float)
            row['median_os_months'] = med if np.isfinite(med) else None
            row['median_ci_95'] = [float(v) if np.isfinite(v) else None for v in mci]
            if time_[m].max() >= t_os:
                ci = kmf.confidence_interval_
                at = ci.loc[ci.index <= t_os].iloc[-1]
                row['os_5y'] = float(kmf.survival_function_at_times(t_os).iloc[0])
                row['os_5y_ci_95'] = [float(at.iloc[0]), float(at.iloc[1])]
        rows.append(row)
    return rows

def boot_c(time_, event, score, boot_idx):
    cs, failed = [], 0
    for b in boot_idx:
        try:
            cs.append(harrell_c(time_[b], event[b], score[b]))
        except Exception:          # a resample with no comparable pair
            failed += 1
    cs = np.array(cs)
    return [float(np.percentile(cs, 2.5)), float(np.percentile(cs, 97.5))], {'resamples': int(len(boot_idx)),
            'failed': failed, 'sd': float(cs.std(ddof=1))}

def evaluate(time_, event, lp, *, cutoff, tau, boot_idx, z=None, ph_check=True):
    """Metrics of plan §8.2 except the calibration slope (§10.4), for one score (higher = higher risk)."""
    time_, event, lp = np.asarray(time_, dtype=float), np.asarray(event, dtype=int), np.asarray(lp, dtype=float)
    if z is None:
        z = (lp - lp.mean()) / lp.std(ddof=1)
    out = {'n': int(len(lp)), 'deaths': int(event.sum()), 'lp_mean': float(lp.mean()), 'lp_sd': float(lp.std(ddof=1))}
    out['hr_per_sd'], cph_sd, df_sd = cox_uni(time_, event, z, 'lp_per_sd')
    out['harrell_c'] = harrell_c(time_, event, lp)
    out['harrell_c_ci_95'], out['harrell_c_bootstrap'] = boot_c(time_, event, lp, boot_idx)
    out['uno_tau_months'] = float(tau)
    out['uno_c'] = uno_c(time_, event, lp, tau)
    high = lp > cutoff
    out['cutoff'] = float(cutoff)
    out['groups'] = km_rows(time_, event, np.where(high, 'high', 'low'), ['high', 'low'])
    out['logrank'], out['hr_high_vs_low'] = None, None
    if 0 < high.sum() < len(high):
        lr = logrank_test(time_[high], time_[~high], event[high], event[~high])
        out['logrank'] = {'chi2': float(lr.test_statistic), 'p': float(lr.p_value)}
        out['hr_high_vs_low'] = cox_uni(time_, event, high.astype(float), 'high_risk')[0]
    if ph_check:
        out['ph_test'] = {}
        for tt in ('rank', 'km'):
            r = proportional_hazard_test(cph_sd, df_sd, time_transform=tt)
            out['ph_test'][tt] = {'statistic': float(r.summary['test_statistic'].iloc[0]),
                                  'p': float(r.summary['p'].iloc[0])}
    return out

def reverse_km_median(time_, event):
    return float(KaplanMeierFitter().fit(time_, 1 - np.asarray(event)).median_survival_time_)

def fmt_p(p):
    if p is None or not np.isfinite(p):
        return 'n/a'
    return f'{p:.1e}' if p < 0.001 else f'{p:.3f}'

def fmt_hr(r):
    return 'n/a' if r is None else f"{r['hr']:.2f} ({r['hr_ci_95'][0]:.2f}–{r['hr_ci_95'][1]:.2f}), p = {fmt_p(r['p_wald'])}"

def fmt_c(est, ci=None):
    return f'{est:.3f}' + ('' if ci is None else f' ({ci[0]:.3f}–{ci[1]:.3f})')

def fmt_os(row):
    if row['os_5y'] is None:
        return 'n/a (follow-up < 60 months)'
    lo, hi = row['os_5y_ci_95']
    return f"{100 * row['os_5y']:.1f}% ({100 * lo:.1f}–{100 * hi:.1f})"

def fmt_median(row):
    m, (lo, hi) = row['median_os_months'], row['median_ci_95']
    if m is None:
        return 'not reached'
    return f"{m:.1f} ({'?' if lo is None else f'{lo:.1f}'}–{'no upper limit' if hi is None else f'{hi:.1f}'})"

def as_bool(s):
    if s.dtype == bool:
        return s
    m = {'true': True, 'false': False, '1': True, '0': False, '1.0': True, '0.0': False}
    out = s.astype(str).str.strip().str.lower().map(m)
    assert out.notna().all(), f'non-boolean values in {s.name}'
    return out.astype(bool)

def entrez_str(idx):
    """Parquet index -> Entrez ID text ('1234.0' -> '1234')."""
    out = []
    for v in idx:
        s = str(v).strip()
        out.append(s[:-2] if s.endswith('.0') else s)
    return pd.Index(out)

print('validation functions defined')

## 6. Gate: open the external cohorts (first time)
Every check above has passed, so the external cohorts are opened now and the opening is logged in `external_access_log_06.jsonl`. For each cohort: the clinical rows (other cohorts are dropped right after loading), the QC flags and the expression file. Patient and death counts before QC must equal the locked plan, and the number of primary QC exclusions must equal the expected number.

TR: Kontrollerin hepsi geçti; dış kohortlar şimdi ilk kez açılıyor ve bu açılış kayda geçiyor. Hasta ve ölüm sayıları ile QC dışlama sayısı kilitli planla aynı olmalı; değilse not defteri durur.

In [ ]:
OPENED_UTC = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
with open(ACCESS_LOG, 'a') as f:
    f.write(json.dumps({'opened_utc': OPENED_UTC, 'code_version': CODE_VERSION, 'model_sha256': MODEL_SHA,
                        'lock_sha256': LOCK_SHA, 'files': [c['file'] for c in COHORTS.values()]}) + '\n')

clin_all = pd.read_csv(f"{PROJECT_DIR}/{LOCK['input_files']['clinical_all']}")
qc_all = pd.read_csv(f"{PROJECT_DIR}/{LOCK['input_files']['qc_samples']}")
CLIN, QCF = {}, {}
for gse in COHORTS:
    CLIN[gse] = clin_all.loc[clin_all['cohort'] == gse].copy()
    QCF[gse] = qc_all.loc[qc_all['sample_id'].isin(CLIN[gse]['sample_id'])].copy()
del clin_all, qc_all                       # GSE181063 rows are not used in this step

D = {}
for gse, c in COHORTS.items():
    cl = CLIN[gse].sort_values('sample_id').reset_index(drop=True)
    assert cl['sample_id'].is_unique
    assert (len(cl), int(cl[EVENT].sum())) == (c['plan']['patients'], c['plan']['deaths']), (
        f"{gse}: {len(cl)} patients / {int(cl[EVENT].sum())} deaths, plan says "
        f"{c['plan']['patients']} / {c['plan']['deaths']}. Stop.")
    qc = QCF[gse].set_index('sample_id').reindex(cl['sample_id'])
    assert qc.index.notna().all() and not qc[[FLAG_P, FLAG_S]].isna().any().any(), f'{gse}: QC flags missing'
    excl = as_bool(qc[FLAG_P]).to_numpy()
    corr = as_bool(qc[FLAG_S]).to_numpy()
    assert int(excl.sum()) == c['expected_excl'], (
        f"{gse}: {int(excl.sum())} primary QC exclusions, plan expects {c['expected_excl']}. Stop.")

    ex = pd.read_parquet(f"{PROJECT_DIR}/{c['file']}")
    if 'gene' in ex.columns:
        ex = ex.set_index('gene')
    ex.index = entrez_str(ex.index)
    assert ex.index.is_unique, f'{gse}: duplicated Entrez IDs in the expression file'
    missing_samples = sorted(set(cl['sample_id']) - set(ex.columns))
    assert not missing_samples, f'{gse}: {len(missing_samples)} clinical samples have no expression column'
    missing_genes = [e for e in G['entrez'] if e not in ex.index]
    assert not missing_genes, f'{gse}: frozen genes not found: {missing_genes}. Stop (no gene may be dropped).'

    keep = ~excl
    X = ex.loc[G['entrez'], cl['sample_id']].to_numpy(dtype=float).T       # samples x 44 genes, model order
    assert np.isfinite(X).all(), f'{gse}: missing expression values in the frozen genes'
    D[gse] = {'clin': cl.loc[keep].reset_index(drop=True), 'X': X[keep], 'corr': corr[keep],
              'excluded': cl.loc[excl, 'sample_id'].tolist(), 'n_before': len(cl), 'deaths_before': int(cl[EVENT].sum()),
              'n_genes_in_file': int(ex.shape[0]), 'n_samples_in_file': int(ex.shape[1])}
    del ex
    d = D[gse]
    print(f"{gse} ({c['role']}): {d['n_before']} patients / {d['deaths_before']} deaths -> primary QC excludes "
          f"{len(d['excluded'])} -> {len(d['clin'])} patients / {int(d['clin'][EVENT].sum())} deaths; "
          f"{int(d['corr'].sum())} correlation outliers kept; 44/44 frozen genes found "
          f"(file: {d['n_genes_in_file']} genes x {d['n_samples_in_file']} samples)")

## 7. Apply the frozen model
Within each cohort, each of the 44 genes is z-scored with the mean and SD (ddof = 1) of that cohort's primary analysis population, and LP = Σ β_g · z_g with the **frozen** coefficients. The training means and SDs in `model_04.json` are not used here: they belong to the DASL platform (plan §10.2). The table compares the spread of the raw expression of the 44 genes between platforms; it is descriptive.

TR: Her kohortta 44 genin her biri o kohortun birincil popülasyonunun ortalaması ve SD'siyle z-skorlanır; risk skoru dondurulmuş katsayılarla hesaplanır. Eğitimdeki ortalama ve SD'ler burada kullanılmaz (farklı platform). Tablo yalnızca betimleyici.

In [ ]:
SCALE = {}
for gse, d in D.items():
    mu = d['X'].mean(axis=0)
    sd = d['X'].std(axis=0, ddof=DDOF)
    zero_sd = G.loc[sd <= 0, 'symbol'].tolist()
    assert not zero_sd, f'{gse}: genes with zero variance in the cohort: {zero_sd}. Stop and check.'
    Z = (d['X'] - mu) / sd
    d['lp'] = Z @ COEF
    SCALE[gse] = {'mean': mu, 'sd': sd}
    print(f"{gse}: LP mean {d['lp'].mean():+.2e} (0 by construction), SD {d['lp'].std(ddof=1):.3f}, "
          f"range {d['lp'].min():.2f} to {d['lp'].max():.2f}")

spread = pd.DataFrame({'symbol': G['symbol'], 'coef': COEF.round(4), 'train_sd_DASL': G['train_sd'].round(3),
                       **{f'sd_{gse}': SCALE[gse]['sd'].round(3) for gse in D}})
print('\nRaw-expression SD of the frozen genes (descriptive; top 10 by |coef|):')
print(spread.reindex(spread['coef'].abs().sort_values(ascending=False).index).head(10).to_string(index=False))

## 8. Primary analysis: H2 and the metrics of §8.2
Per cohort: HR per 1 SD of LP (H2), Harrell C with 1,000-resample bootstrap CI, Uno C (τ rule of §8.2), Kaplan–Meier high vs low risk at the cohort's own median LP, log-rank test, HR high vs low and the Schoenfeld check. Tertiles (the cohort's own tertiles) are descriptive. The bootstrap index vectors are drawn with the plan seed, as in step 5.

TR: Asıl sonuç. Her kohort için H2 (SD başına HR > 1 ve %95 GA 1'i dışlıyor), Harrell C (bootstrap GA), Uno C, kohortun kendi medyanıyla KM + log-rank, yüksek/düşük HR ve orantılı riskler testi. Tertiller yalnızca betimleyici. GSE10846'nın takibi kısa (medyan ~34 ay); bu yüzden Uno C için τ büyük olasılıkla 60 aydan kısa olacak ve 5 yıllık OS bazı gruplarda hesaplanamayabilir.

In [ ]:
PRIMARY, TERTILES, H2 = {}, {}, {}
TERT_LABELS = ['T1 (low)', 'T2', 'T3 (high)']
for gse, d in D.items():
    T, E, LP = d['clin'][TIME].to_numpy(dtype=float), d['clin'][EVENT].to_numpy(dtype=int), d['lp']
    d['T'], d['E'] = T, E
    d['cutoff'] = float(np.median(LP))
    d['tau'], d['tau_rule'] = uno_tau(T, TAU0, MIN_AT_RISK)
    d['z'] = (LP - LP.mean()) / LP.std(ddof=1)
    boot = np.random.default_rng(BOOT_SEED).integers(0, len(T), size=(N_BOOT, len(T)))
    R = evaluate(T, E, LP, cutoff=d['cutoff'], tau=d['tau'], boot_idx=boot, z=d['z'])
    R['uno_tau_rule'] = d['tau_rule']
    R['median_follow_up_months_reverse_km'] = reverse_km_median(T, E)
    PRIMARY[gse] = R
    q1, q2 = np.quantile(LP, [1 / 3, 2 / 3])
    d['tertile_cutoffs'] = [float(q1), float(q2)]
    d['tertile'] = np.where(LP <= q1, TERT_LABELS[0], np.where(LP <= q2, TERT_LABELS[1], TERT_LABELS[2]))
    TERTILES[gse] = km_rows(T, E, d['tertile'], TERT_LABELS)
    h = R['hr_per_sd']
    H2[gse] = bool(h['hr'] > 1 and h['hr_ci_95'][0] > 1)

    print(f"\n=== {gse} ({COHORTS[gse]['role']}) — n = {R['n']}, deaths = {R['deaths']}, "
          f"median follow-up {R['median_follow_up_months_reverse_km']:.1f} months")
    print(f"H2  HR per SD of LP: {fmt_hr(h)}  ->  {'MET' if H2[gse] else 'NOT MET'}")
    print(f"Harrell C: {fmt_c(R['harrell_c'], R['harrell_c_ci_95'])}   Uno C (tau = {d['tau_rule']}): {R['uno_c']:.3f}")
    for row in R['groups']:
        print(f"  {row['group']:5s} risk: n = {row['n']:3d}, deaths = {row['deaths']:3d}, median OS {fmt_median(row)} months, "
              f'5-year OS {fmt_os(row)}')
    lr = R['logrank']
    print(f"Log-rank (own median {d['cutoff']:.4f}): chi2 = {lr['chi2']:.2f}, p = {fmt_p(lr['p'])}; "
          f"HR high vs low: {fmt_hr(R['hr_high_vs_low'])}")
    print(f"Proportional hazards (Schoenfeld, descriptive): p = {fmt_p(R['ph_test']['rank']['p'])} (rank), "
          f"{fmt_p(R['ph_test']['km']['p'])} (KM)")
    print('Tertiles (descriptive): ' + '; '.join(
        f"{r['group']} n {r['n']}, {r['deaths']} deaths, 5-y OS {fmt_os(r)}" for r in TERTILES[gse]))

## 9. Sensitivity analyses (plan §10.6)
Both use the LP values of the primary analysis, its per-SD scaling, its median cutoff and its τ; nothing is re-scaled (`sensitivity_scaling` in the plan). Bootstrap with the plan seed on the subset size, as in step 5.

1. GSE87371 without R-mini-CHOP21 patients (plan: n = 13).
2. Correlation outliers excluded, in each cohort.

TR: İki duyarlılık analizi, birincil analizin skorlarıyla, yeniden ölçekleme yapmadan: GSE87371'de R-mini-CHOP21 hastaları hariç ve her kohortta korelasyon aykırıları hariç.

In [ ]:
def subset_eval(gse, mask, label):
    d = D[gse]
    boot = np.random.default_rng(BOOT_SEED).integers(0, int(mask.sum()), size=(N_BOOT, int(mask.sum())))
    R = evaluate(d['T'][mask], d['E'][mask], d['lp'][mask], cutoff=d['cutoff'], tau=d['tau'], boot_idx=boot,
                 z=d['z'][mask], ph_check=False)
    return {'cohort': gse, 'analysis': label, **R}

SENS = []
for gse in D:
    P = PRIMARY[gse]
    SENS.append({'cohort': gse, 'analysis': 'primary', **{k: P[k] for k in P if k not in ('ph_test',)}})

treat = D['GSE87371']['clin']['treatment'].astype(str)
MINI = treat.str.lower().str.replace(r'[\s_]', '-', regex=True).str.contains('mini-chop').to_numpy()
N_MINI = int(MINI.sum())
print(f'GSE87371 treatment groups: {treat.value_counts().to_dict()}')
SENS.append(subset_eval('GSE87371', ~MINI, f'without R-mini-CHOP21 (n excluded = {N_MINI})'))
for gse in D:
    SENS.append(subset_eval(gse, ~D[gse]['corr'], f"correlation outliers excluded (n excluded = {int(D[gse]['corr'].sum())})"))

SENS_EXTRA_WARNINGS = []
if N_MINI != MINI_CHOP_EXPECTED:
    SENS_EXTRA_WARNINGS.append(f'R-mini-CHOP21 patients: {N_MINI}, plan says {MINI_CHOP_EXPECTED}')

def sens_row(r):
    return {'cohort': r['cohort'], 'analysis': r['analysis'], 'n': r['n'], 'deaths': r['deaths'],
            'hr_per_sd': r['hr_per_sd']['hr'], 'hr_ci_low': r['hr_per_sd']['hr_ci_95'][0],
            'hr_ci_high': r['hr_per_sd']['hr_ci_95'][1], 'p_wald': r['hr_per_sd']['p_wald'],
            'harrell_c': r['harrell_c'], 'c_ci_low': r['harrell_c_ci_95'][0], 'c_ci_high': r['harrell_c_ci_95'][1],
            'uno_c': r['uno_c'], 'uno_tau_months': r['uno_tau_months'], 'cutoff': r['cutoff'],
            'logrank_p': None if r['logrank'] is None else r['logrank']['p'],
            'hr_high_vs_low': None if r['hr_high_vs_low'] is None else r['hr_high_vs_low']['hr']}

SENS_ROWS = [sens_row(r) for r in SENS]
SENS_DF = pd.DataFrame(SENS_ROWS)
with pd.option_context('display.width', 200, 'display.max_columns', 30):
    print(SENS_DF[['cohort', 'analysis', 'n', 'deaths', 'hr_per_sd', 'hr_ci_low', 'hr_ci_high', 'p_wald',
                   'harrell_c', 'c_ci_low', 'c_ci_high', 'uno_c', 'logrank_p']].round(4).to_string(index=False))
for w in SENS_EXTRA_WARNINGS:
    print('WARNING:', w)

## 10. Figures
Kaplan–Meier curves (own median and own tertiles) for each cohort, the LP distributions, and a forest plot of the HR per SD in the internal test set (from `manifest_05.json`), the two external cohorts and the sensitivity analyses. Saved in `figures/06/`.

TR: Her kohort için KM eğrileri (medyan ve tertil), LP dağılımları ve SD başına HR'leri yan yana gösteren bir orman grafiği.

In [ ]:
BLUE, ORANGE, AQUA, INK, MUTED, GRID = '#2a78d6', '#eb6834', '#1baf7a', '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({'axes.edgecolor': MUTED, 'axes.labelcolor': INK, 'xtick.color': MUTED, 'ytick.color': MUTED,
                     'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})

def km_figure(gse, labels, order, colors, title, path):
    d = D[gse]
    fig, ax = plt.subplots(figsize=(7, 5))
    fitters, names = [], []
    for g, col in zip(order, colors):
        m = labels == g
        kmf = KaplanMeierFitter(label=g).fit(d['T'][m], d['E'][m])
        kmf.plot_survival_function(ax=ax, ci_show=True, color=col, lw=2, ci_alpha=0.12)
        fitters.append(kmf)
        names.append(f"{g} (n = {int(m.sum())}, deaths = {int(d['E'][m].sum())})")
    ax.set_xlabel('Months'); ax.set_ylabel('Overall survival'); ax.set_ylim(0, 1.02)
    ax.grid(axis='y', color=GRID, lw=0.8); ax.set_title(title, loc='left', color=INK, fontsize=10)
    handles = [h for h in ax.get_legend_handles_labels()[0] if not isinstance(h, matplotlib.collections.PolyCollection)]
    ax.legend(handles[:len(names)], names, frameon=False, loc='lower left')
    add_at_risk_counts(*fitters, ax=ax, rows_to_show=['At risk'])
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.show(); plt.close(fig)

FIG_FILES = []
for gse, d in D.items():
    P = PRIMARY[gse]
    p1 = f'{FIG}/km_{gse}_median_06.png'
    km_figure(gse, np.where(d['lp'] > d['cutoff'], 'high', 'low'), ['low', 'high'], [BLUE, ORANGE],
              f"{gse}: own median split (log-rank p = {fmt_p(P['logrank']['p'])})", p1)
    p2 = f'{FIG}/km_{gse}_tertiles_06.png'
    km_figure(gse, d['tertile'], TERT_LABELS, [BLUE, AQUA, ORANGE], f'{gse}: own tertiles (descriptive)', p2)
    FIG_FILES += [p1, p2]

fig, axes = plt.subplots(1, len(D), figsize=(10, 3.6), sharey=False)
for ax, (gse, d) in zip(np.atleast_1d(axes), D.items()):
    ax.hist(d['lp'][d['E'] == 0], bins=25, color=BLUE, alpha=0.75, label='censored', edgecolor='white', lw=0.5)
    ax.hist(d['lp'][d['E'] == 1], bins=25, color=ORANGE, alpha=0.75, label='died', edgecolor='white', lw=0.5)
    ax.axvline(d['cutoff'], color=INK, lw=1, ls='--'); ax.set_title(gse, loc='left'); ax.set_xlabel('LP (within-cohort z)')
    ax.legend(frameon=False)
fig.tight_layout(); p3 = f'{FIG}/lp_distribution_06.png'; fig.savefig(p3, dpi=150); plt.show(); plt.close(fig)
FIG_FILES.append(p3)

rows = [('GSE181063 test (step 5)', TEST_RESULT['hr_per_sd'], *TEST_RESULT['hr_per_sd_ci_95'], 162)]
for r in SENS_ROWS:
    tag = '' if r['analysis'] == 'primary' else '   ' + r['analysis'].split(' (')[0]
    rows.append((f"{r['cohort']}{tag}", r['hr_per_sd'], r['hr_ci_low'], r['hr_ci_high'], r['n']))
fig, ax = plt.subplots(figsize=(8, 0.45 * len(rows) + 1.2))
for i, (lab, hr, lo, hi, n) in enumerate(rows[::-1]):
    primary = not lab.startswith(tuple(f'{g}   ' for g in D))
    ax.plot([lo, hi], [i, i], color=INK if primary else MUTED, lw=2, solid_capstyle='round')
    ax.plot(hr, i, 'o', ms=8 if primary else 6, color=BLUE if primary else MUTED)
    ax.text(hi * 1.04, i, f'{hr:.2f} ({lo:.2f}–{hi:.2f})', va='center', color=MUTED, fontsize=9)
ax.axvline(1, color=MUTED, lw=1, ls='--')
ax.set_xscale('log')
ticks = [t for t in (0.5, 0.75, 1, 1.5, 2, 3, 4) if min(r[2] for r in rows) / 1.3 <= t <= max(r[3] for r in rows) * 1.3]
ax.set_xticks(ticks); ax.set_xticklabels([f'{t:g}' for t in ticks]); ax.minorticks_off()
ax.set_xlim(min(min(r[2] for r in rows), 0.9) / 1.1, max(r[3] for r in rows) * 1.6)
ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[0] for r in rows[::-1]])
ax.set_xlabel('HR per SD of LP (95% CI, log scale)'); ax.grid(axis='x', color=GRID, lw=0.8)
ax.set_title('Frozen 44-gene score: internal and external validation', loc='left', color=INK)
fig.tight_layout(); p4 = f'{FIG}/forest_hr_per_sd_06.png'; fig.savefig(p4, dpi=150); plt.show(); plt.close(fig)
FIG_FILES.append(p4)

## 11. Red flags and the pre-specified conclusion
Red flag (plan §13): any C-index > 0.90. The test-set rule (C more than 0.10 above the nested-CV mean) is defined for the internal test set; here the same gap is reported as an extra warning only. Conclusion wording is fixed by plan §12: "Externally validated" = H2 met in GSE10846 **and** HR > 1 in GSE87371.

TR: Kırmızı bayrak kontrolü ve planın 12. bölümündeki önceden sabitlenmiş sonuç cümlesi.

In [ ]:
RED_FLAGS, EXTRA_WARNINGS = [], list(SENS_EXTRA_WARNINGS)
for r in SENS_ROWS:
    for key in ('harrell_c', 'uno_c'):
        if r[key] is not None and r[key] > 0.90:
            RED_FLAGS.append(f"{r['cohort']} / {r['analysis']}: {key} = {r[key]:.3f} > 0.90")
C_GAP = {gse: PRIMARY[gse]['harrell_c'] - NESTED_CV_C for gse in D}
for gse, g in C_GAP.items():
    if g > 0.10:
        EXTRA_WARNINGS.append(f'{gse}: C exceeds the nested-CV mean C by {g:.3f} (> 0.10)')

HR87 = PRIMARY['GSE87371']['hr_per_sd']['hr']
EXT_VALIDATED = bool(H2['GSE10846'] and HR87 > 1)
if EXT_VALIDATED:
    CONCLUSION = 'Externally validated (H2 met in GSE10846 and HR > 1 in GSE87371)'
else:
    parts = [f"H2 in GSE10846 {'met' if H2['GSE10846'] else 'not met'}",
             f"HR in GSE87371 = {HR87:.2f} ({'> 1' if HR87 > 1 else '<= 1'})"]
    CONCLUSION = 'Not externally validated per plan §12 (' + '; '.join(parts) + ')'
SUPPORT = {gse: {'Harrell C 95% CI above 0.5': bool(PRIMARY[gse]['harrell_c_ci_95'][0] > 0.5),
                 'log-rank p < 0.05': bool(PRIMARY[gse]['logrank']['p'] < 0.05)} for gse in D}

print('Red flags:', RED_FLAGS if RED_FLAGS else 'none')
print('Extra warnings:', EXTRA_WARNINGS if EXTRA_WARNINGS else 'none')
for gse in D:
    print(f"{gse}: H2 {'met' if H2[gse] else 'not met'}; C − nested-CV C = {C_GAP[gse]:+.3f}; supporting: {SUPPORT[gse]}")
print('\nCONCLUSION (plan §12):', CONCLUSION)
if RED_FLAGS:
    print('\nSTOP: a red flag was raised. Look for an error before reporting anything.')

## 12. Save the results, the manifest and the summary
Patient-level scores (`external_lp_06.csv`, used again in step 7), all metrics, figures and `manifest_06.json` with the SHA-256 of every input and output. The key results of the first complete run are written once to `first_results_06.json` and never overwritten; a later run is compared with it (plan §17).

TR: Bütün çıktıları `results/06/` ve `figures/06/` klasörlerine yazar. İlk tam çalıştırmanın ana sonuçları `first_results_06.json` dosyasına bir kez yazılır ve bir daha değiştirilmez. En alttaki özeti bana yapıştır.

In [ ]:
lp_rows = []
for gse, d in D.items():
    lp_rows.append(pd.DataFrame({
        'cohort': gse, 'sample_id': d['clin']['sample_id'], 'lp': d['lp'], 'lp_per_sd': d['z'],
        'risk_group': np.where(d['lp'] > d['cutoff'], 'high', 'low'), 'tertile': d['tertile'], FLAG_S: d['corr'],
        'r_mini_chop21': MINI if gse == 'GSE87371' else False}))
pd.concat(lp_rows, ignore_index=True).to_csv(f'{RES}/external_lp_06.csv', index=False)
pd.DataFrame([{'cohort': gse, 'grouping': 'own median', **r} for gse in D for r in PRIMARY[gse]['groups']]
             + [{'cohort': gse, 'grouping': 'own tertiles (descriptive)', **r} for gse in D for r in TERTILES[gse]]
             ).to_csv(f'{RES}/km_external_06.csv', index=False)
pd.DataFrame({'entrez': G['entrez'], 'symbol': G['symbol'], 'coef': COEF,
              **{f'mean_{g}': SCALE[g]['mean'] for g in D}, **{f'sd_{g}': SCALE[g]['sd'] for g in D}}
             ).to_csv(f'{RES}/cohort_scaling_06.csv', index=False)
SENS_DF.to_csv(f'{RES}/sensitivity_06.csv', index=False)
write_json(SENS, f'{RES}/sensitivity_06.json')
VALIDATION = {
    'populations': {gse: {'patients_before_qc': D[gse]['n_before'], 'deaths_before_qc': D[gse]['deaths_before'],
                          'excluded_by_primary_qc': D[gse]['excluded'], 'patients': PRIMARY[gse]['n'],
                          'deaths': PRIMARY[gse]['deaths'], 'corr_outliers_kept': int(D[gse]['corr'].sum())} for gse in D},
    'scaling': 'gene-wise z-score within each cohort (primary population, ddof 1); frozen coefficients',
    'primary': PRIMARY, 'tertiles_descriptive': TERTILES,
    'tertile_cutoffs': {gse: D[gse]['tertile_cutoffs'] for gse in D},
    'H2': {'definition': S6['H2'], 'met': H2}, 'supporting_evidence': SUPPORT, 'red_flags': RED_FLAGS,
    'extra_warnings': EXTRA_WARNINGS, 'c_minus_nested_cv_mean_c': C_GAP, 'conclusion': CONCLUSION}
write_json(VALIDATION, f'{RES}/external_validation_06.json')

KEY = {'code_version': CODE_VERSION, 'model_sha256': MODEL_SHA}
for gse in D:
    P = PRIMARY[gse]
    KEY[gse] = {'n': P['n'], 'deaths': P['deaths'], 'hr_per_sd': P['hr_per_sd']['hr'], 'hr_per_sd_ci_95': P['hr_per_sd']['hr_ci_95'],
                'p_wald': P['hr_per_sd']['p_wald'], 'harrell_c': P['harrell_c'], 'harrell_c_ci_95': P['harrell_c_ci_95'],
                'uno_c': P['uno_c'], 'uno_tau_months': P['uno_tau_months'], 'logrank_p': P['logrank']['p'], 'h2_met': H2[gse]}
KEY['externally_validated'] = EXT_VALIDATED

def _same(a, b):
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    if isinstance(a, list) and isinstance(b, list):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, float) or isinstance(b, float):
        return a is not None and b is not None and bool(np.isclose(a, b, rtol=1e-9, atol=1e-12))
    return a == b

KEY_CLEAN = _clean(KEY)
if not os.path.exists(FIRST_PATH):
    write_json({'note': 'Written once, at the first complete run of step 6; never overwritten (plan §17).',
                'written_utc': OPENED_UTC, **KEY_CLEAN}, FIRST_PATH)
    FIRST_STATUS = 'first complete run: first_results_06.json written'
else:
    with open(FIRST_PATH) as f:
        FIRST = json.load(f)
    same = all(_same(KEY_CLEAN[k], FIRST.get(k)) for k in KEY_CLEAN)
    FIRST_STATUS = ('re-run: identical to first_results_06.json' if same else
                    're-run: DIFFERS from first_results_06.json — report both (plan §17) and find out why')
print(FIRST_STATUS)

SUMMARY = [f'Step 6 external validation ({CODE_VERSION}) | model {MODEL_SHA[:12]} | lock {LOCK_SHA[:12]} | {FIRST_STATUS}']
for gse in D:
    P = PRIMARY[gse]
    SUMMARY.append(f"{gse} ({COHORTS[gse]['role']}): n {P['n']}, deaths {P['deaths']} (QC excluded {len(D[gse]['excluded'])}); "
                   f"median FU {P['median_follow_up_months_reverse_km']:.1f} mo")
    SUMMARY.append(f"  H2 HR/SD {fmt_hr(P['hr_per_sd'])} -> {'MET' if H2[gse] else 'NOT MET'}")
    SUMMARY.append(f"  Harrell C {fmt_c(P['harrell_c'], P['harrell_c_ci_95'])}; Uno C {P['uno_c']:.3f} (tau {P['uno_tau_rule']})")
    SUMMARY.append(f"  log-rank p {fmt_p(P['logrank']['p'])}; HR high/low {fmt_hr(P['hr_high_vs_low'])}; "
                   f"PH p {fmt_p(P['ph_test']['rank']['p'])} (rank)")
    for row in P['groups']:
        SUMMARY.append(f"  {row['group']}: n {row['n']}, deaths {row['deaths']}, median OS {fmt_median(row)}, 5-y OS {fmt_os(row)}")
    SUMMARY.append('  tertiles: ' + '; '.join(f"{r['group']} n {r['n']}/{r['deaths']} d, 5-y {fmt_os(r)}" for r in TERTILES[gse]))
SUMMARY.append('Sensitivity (primary scores, no re-scaling):')
for r in SENS_ROWS:
    if r['analysis'] != 'primary':
        SUMMARY.append(f"  {r['cohort']} {r['analysis']}: n {r['n']}, deaths {r['deaths']}, HR/SD {r['hr_per_sd']:.2f} "
                       f"({r['hr_ci_low']:.2f}–{r['hr_ci_high']:.2f}), C {r['harrell_c']:.3f}, log-rank p {fmt_p(r['logrank_p'])}")
SUMMARY.append(f"Red flags: {RED_FLAGS or 'none'} | extra warnings: {EXTRA_WARNINGS or 'none'}")
SUMMARY.append(f'CONCLUSION: {CONCLUSION}')
with open(f'{RES}/summary_06.txt', 'w') as f:
    f.write('\n'.join(SUMMARY) + '\n')

OUT_FILES = ['external_lp_06.csv', 'km_external_06.csv', 'cohort_scaling_06.csv', 'sensitivity_06.csv',
             'sensitivity_06.json', 'external_validation_06.json', 'first_results_06.json', 'summary_06.txt',
             'external_access_log_06.jsonl']
MANIFEST = {
    'step': '06', 'notebook': '06_external_validation.ipynb', 'code_version': CODE_VERSION,
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'plan': {'version': PLAN['version'], 'plan_json_sha256': LOCK['plan_json_sha256'], 'plan_md_sha256': LOCK['plan_md_sha256'],
             'lock_file_sha256': LOCK_SHA, 'amendments': len(AMENDMENTS), 'sections_implemented': ['10', '12', '13']},
    'frozen_model': {'file': 'results/04/model_04.json', 'sha256': MODEL_SHA, 'matches': list(HASH_REFS),
                     'n_genes': int(len(G)), 'lambda': MODEL['penalty']['lambda']},
    'inputs_verified_against_lock': INPUT_SHA,
    'data_access': {'gates_before_opening': ['plan lock, plan files and amendments verified', 'all input files match lock, manifest_04b and manifest_05',
                                             'frozen model SHA-256 matches 4 records', 'manifest_05: external files never opened in step 5'],
                    'external_opened_utc': OPENED_UTC, 'access_log': 'results/06/external_access_log_06.jsonl',
                    'openings_logged': PREVIOUS_OPENINGS + 1,
                    'opened': ['GSE10846 and GSE87371 rows of data/clean/clinical_clean_all.csv',
                               'GSE10846 and GSE87371 rows of data/processed/qc_samples.csv',
                               'data/processed/expr_GSE10846.parquet', 'data/processed/expr_GSE87371.parquet',
                               'results/04/model_04.json, manifest_04b.json; results/05/manifest_05.json'],
                    'not_opened': ['GSE181063 expression files (training and test)']},
    'populations': VALIDATION['populations'],
    'external_validation': KEY_CLEAN, 'H2_met': H2, 'supporting_evidence': SUPPORT,
    'sensitivity': SENS_ROWS, 'conclusion': CONCLUSION, 'red_flags': RED_FLAGS, 'extra_warnings': EXTRA_WARNINGS,
    'c_minus_nested_cv_mean_c': C_GAP, 'first_results_status': FIRST_STATUS,
    'implementation_notes': [
        'Order: plan lock and inputs (vs lock, manifest_04b, manifest_05); frozen model hash (4 records) and content; step-5 state; external cohorts opened (logged).',
        'Scoring: the 44 frozen Entrez IDs (parquet index converted to text, "1234.0" -> "1234"); each gene z-scored with the mean and SD (ddof 1) of the cohort primary population; LP = sum of frozen coef x z. Training means/SDs of model_04.json not used.',
        'Risk groups: high = LP > median LP of the cohort primary population; tertiles from the cohort own LP tertiles (T1: LP <= q1; T2: q1 < LP <= q2; T3: LP > q2), descriptive.',
        "Cox models: lifelines CoxPHFitter (Efron ties, no penalty). HR per SD: LP standardized with the mean and SD (ddof 1) of the cohort primary population. No calibration slope (plan 10.4). PH check: lifelines proportional_hazard_test, rank and KM transforms.",
        'Harrell C: scikit-survival concordance_index_censored; 1,000 bootstrap resamples, numpy default_rng(20261415).integers, all index vectors drawn at once per cohort; percentile interval.',
        'Uno C: concordance_index_ipcw with the cohort primary population as censoring and evaluation sample; tau = 60 months unless fewer than 20% have follow-up >= 60 months, then the latest time with >= 20% at risk.',
        'Kaplan-Meier: lifelines defaults; 5-year OS = KM at 60 months, reported only if the group has follow-up >= 60 months.',
        'Sensitivity, not fixed in detail by the plan: LP values, per-SD scaling, median cutoff and tau of the primary analysis; bootstrap default_rng(20261415) on the subset size. R-mini-CHOP21 identified from the clinical treatment column (text contains "mini-chop").',
        'Red flags: any Harrell or Uno C > 0.90. The test-set gap rule of plan 13 is reported for the external cohorts as an extra warning only.'],
    'environment': {'python': platform.python_version(), 'platform': platform.platform(), 'cpu_count': os.cpu_count(),
                    'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__,
                    'scikit-learn': sklearn.__version__, 'scikit-survival': sksurv.__version__,
                    'lifelines': lifelines.__version__, 'matplotlib': matplotlib.__version__},
    'environment_step5': M5['environment'],
    'runtime_seconds_this_run': round(time.time() - T_START, 1),
    'outputs': {**{f'results/06/{f}': sha256_file(f'{RES}/{f}') for f in OUT_FILES},
                **{os.path.relpath(p, PROJECT_DIR): sha256_file(p) for p in FIG_FILES}},
}
write_json(MANIFEST, f'{RES}/manifest_06.json')
print(f"manifest_06.json written (SHA-256 {sha256_file(f'{RES}/manifest_06.json')[:12]})\n")
print('=' * 30, 'SUMMARY — paste this back', '=' * 30)
print('\n'.join(SUMMARY))